In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_filtered_set;
CREATE TABLE dev.mohit_gangwani.ad_labeling_filtered_set AS
SELECT vc.external_id AS ad_id
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
FROM prod.detection.viewing_commercials_firehose_dedup vc
LEFT JOIN (
  SELECT cief.fk_commercial_id, cief.external_id
  FROM prod.detection.commercial_id_external_firehose cief
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  WHERE cl.client_name = 'kinetiq'
  GROUP BY 1, 2
) cief
  ON cief.external_id = vc.external_id
WHERE vc.session_start >= CURRENT_DATE - 8
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND (cief.external_id IS NOT NULL OR vc.fk_commercial_source_id = 2)
GROUP BY 1, 2
HAVING COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) >= 20
;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_base_viewing_table_new;
CREATE TABLE dev.mohit_gangwani.ad_labeling_base_viewing_table_new AS
SELECT vc.fk_tvid
, vc.session_start
, COALESCE(vc.external_id, cief.ad_id) AS ad_id
, NULLIF(NULLIF(l.iso_state, 'none'), '') AS us_state
-- , CASE WHEN UPPER(COALESCE(vc.prev_content_type, content.content_type)) <=> 'LINEAR'
--          OR UPPER(COALESCE(vc.reported_input_source, content.reported_input_source)) <=> 'ANTENNA'
--          OR COALESCE(vc.prev_station_id, content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'LINEAR'
--        WHEN UPPER(COALESCE(vc.prev_content_type, content.content_type)) <=> 'STREAMING'
--          OR COALESCE(vc.prev_vizio_epg_station, content.vizio_epg_station) IS NOT NULL
--          OR UPPER(COALESCE(vc.reported_input_source, content.reported_input_source)) <=> 'APPS' THEN 'APPS'
--        ELSE 'UNKNOWN' END AS app_or_linear
-- , CASE WHEN st.local_or_national = 'Local' OR COALESCE(content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'Local'
--        WHEN st.local_or_national IS NOT NULL THEN 'National'
--        ELSE 'Unknown' END AS station_type
-- , CASE WHEN content.session_start <= TIMESTAMPADD(SECOND, content.runtime, content.airdate) THEN 'Live'
--        WHEN COALESCE(content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'Timeshifted'
--        WHEN COALESCE(vc.prev_vizio_epg_station, content.vizio_epg_station) IS NOT NULL THEN 'Live'
--        WHEN content.is_live = TRUE THEN 'Live'
--        WHEN content.is_live = FALSE THEN 'Timeshifted'
--   END AS is_live
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
-- , DATE_TRUNC('day', vc.session_start) AS event_date
-- , DATE_TRUNC('HOUR', vc.session_start) AS time_bin_start
-- , DATE_TRUNC('HOUR', vc.session_start) + INTERVAL '1 HOUR' AS time_bin_end
-- , TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(vc.session_start) / 600) * 600) AS time_bin_start
-- , TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(vc.session_start) / 600) * 600 + 600) AS time_bin_end
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN dev.mohit_gangwani.ad_labeling_filtered_set cief
  ON cief.ad_id = vc.external_id
 AND cief.fk_dma_id = NVL(vc.fk_dma_id, 0)
JOIN prod.detection.location l
  ON l.location_id = vc.fk_location_id
 AND l.country_code = 'US'
-- LEFT JOIN prod.detection.viewing_content_firehose content
--   ON vc.fk_tvid = content.fk_tvid
--  AND vc.prev_session_start = content.session_start
--  AND content.session_start >= CURRENT_DATE - 9
-- LEFT JOIN prod.detection.epg_station st
--   ON st.station_id = COALESCE(vc.prev_station_id, content.fk_station_id)
--  AND st.vendor_name = 'TIVO'
WHERE vc.session_start >= CURRENT_DATE - 8
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
GROUP BY ALL;

In [0]:
%sql
-- # Active TVs per DMA × 10-min bin × platform/station
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_opportunities_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_opportunities_unbinned AS
WITH total_count AS (
  SELECT COUNT(DISTINCT fk_tvid) AS ttl_tvs
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new
)
SELECT a.fk_dma_id
-- , app_or_linear
-- , station_type
, COUNT(DISTINCT a.fk_tvid) AS active_tvs
, COUNT(DISTINCT a.fk_tvid||'-'||session_start) AS total_impressions
, active_tvs*1.0/t.ttl_tvs AS dma_perc
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new a, total_count t
GROUP BY 1, t.ttl_tvs

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_opportunities_unbinned

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_impression_count_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_impression_count_unbinned AS
SELECT n.fk_dma_id
-- , n.app_or_linear
-- , n.station_type
, n.ad_id
, COUNT(DISTINCT fk_tvid||'_'||session_start) AS impressions
-- , SUM(CASE WHEN n.is_live = 'Live' THEN 1 ELSE 0 END) AS live_impressions
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new n
GROUP BY 1,2--,3,4;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_ad_impression_count_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_ad_impression_count_unbinned AS
SELECT n.ad_id
, COUNT(DISTINCT us_state) AS total_states
, COUNT(DISTINCT fk_tvid||'_'||session_start) AS total_impressions
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new n
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_perc_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_perc_unbinned AS
WITH ad_dma AS (
  SELECT ad_id
  , fk_dma_id
  , impressions AS impression_count
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
)
, ad_tot AS (
  SELECT ad_id
  , total_impressions
  FROM dev.mohit_gangwani.ad_labeling_ad_impression_count_unbinned
)
, joined AS (
SELECT d.ad_id
, d.fk_dma_id
, d.impression_count
, t.total_impressions
, o.active_tvs
, o.dma_perc
, d.impression_count/t.total_impressions AS per_dma
FROM ad_dma d
JOIN ad_tot t
  ON d.ad_id = t.ad_id
JOIN dev.mohit_gangwani.ad_labeling_opportunities_unbinned o
  ON o.fk_dma_id = d.fk_dma_id
)
SELECT *
FROM (
  SELECT *
  , DENSE_RANK() OVER (PARTITION BY ad_id ORDER BY per_dma DESC) AS dma_rank
  , SUM(per_dma) OVER (PARTITION BY ad_id ORDER BY per_dma DESC) AS cumulative_perc
  FROM joined
)
WHERE dma_rank = 1 OR cumulative_perc <= 0.90

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
WHERE ad_id IN ('AE16546-2025-49-09116', 'AE16546-2024-32-04203', 'AE16546-2025-44-04995', 'AE16546-2025-31-02907', 'AE16546-2025-32-04566', 'AE16546-2024-36-04680', 'AE16546-2025-49-07019', 'AE16546-2025-34-04655')
ORDER BY ad_id, dma_rank

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_phat_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_phat_unbinned AS
SELECT i.fk_dma_id
-- , i.app_or_linear
-- , i.station_type
, i.ad_id
, i.impressions
, COALESCE(o.active_tvs, 0) AS opportunities
, CASE WHEN COALESCE(o.active_tvs,0) > 0 THEN i.impressions / o.active_tvs
       ELSE 0.0
  END AS p_hat_dma
-- , CASE WHEN i.impressions > 0 THEN i.live_impressions / i.impressions END AS live_share
FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned i
LEFT JOIN dev.mohit_gangwani.ad_labeling_opportunities_unbinned o
  ON i.fk_dma_id = o.fk_dma_id
--  AND i.app_or_linear = o.app_or_linear
--  AND i.station_type = o.station_type
GROUP BY ALL;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
ORDER BY impressions DESC
LIMIT 100

In [0]:
%sql
SELECT dma_count, COUNT(DISTINCT ad_id)
FROM (
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1)
WHERE dma_count > 1
GROUP BY 1
ORDER BY 1 DESC

In [0]:
%sql
SELECT AVG(dma_count)
FROM (
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1)
-- WHERE dma_count > 1
-- GROUP BY 1
ORDER BY 1 DESC

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_over_95_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_over_95_unbinned AS
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_in_95
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_coverage_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_coverage_unbinned AS
WITH flags AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
  GROUP BY 1--, 2, 3
)
, ttl AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  -- SELECT COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  -- FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
  -- SELECT AVG(dma_count) AS dma_count
  --   FROM (
  --   SELECT ad_id
  --   , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
    SELECT COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
    FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
    -- GROUP BY 1
  -- )
  -- WHERE dma_count > 1
  -- GROUP BY 1, 2
)
-- SELECT a.app_or_linear
-- , a.station_type
-- , ini_calc AS (
  SELECT a.ad_id
  -- , a.dma_count AS dma_in_95
  -- , a.ad_id
  , a.dma_count/t.dma_count AS coverage_score
  FROM flags a, ttl t
  -- LEFT JOIN ttl t
  --   ON t.app_or_linear = a.app_or_linear
  --  AND t.station_type = a.station_type
  GROUP BY ALL
-- )
-- , mm_calc AS (
--   SELECT MIN(coverage_score_one) AS min_cs
--   , MAX(coverage_score_one) AS max_cs
--   FROM ini_calc
-- )
-- SELECT i.ad_id
-- -- , dma_in_95
-- , (coverage_score_one-min_cs)/(max_cs-min_cs) AS coverage_score
-- FROM ini_calc i, mm_calc;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_coverage_unbinned
WHERE ad_id in ('AE16546-2025-49-09116', 'AE16546-2024-32-04203', 'AE16546-2025-44-04995', 'AE16546-2025-31-02907', 'AE16546-2025-32-04566', 'AE16546-2024-36-04680', 'AE16546-2025-49-07019', 'AE16546-2025-34-04655')
ORDER BY coverage_score
LIMIT 100

In [0]:
%sql
-- Entropy across DMAs in a bin
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_entropy_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_entropy_unbinned AS
WITH base AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
)
, tot AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , SUM(impressions)*1.0 AS tot_imp
  , COUNT(DISTINCT fk_dma_id) AS k_dmas
  FROM base
  GROUP BY 1--,2,3
)
, joined AS (
  -- SELECT b.app_or_linear
  -- , b.station_type
  -- , b.ad_id
  SELECT b.ad_id
  , b.fk_dma_id
  , b.impressions
  , t.tot_imp
  , t.k_dmas
  , CASE WHEN t.tot_imp>0 THEN b.impressions / t.tot_imp ELSE 0.0 END AS p_dma
  FROM base b
  JOIN tot t 
  --   ON b.app_or_linear = t.app_or_linear
  --  AND b.station_type = t.station_type
  --  AND b.ad_id = t.ad_id
    ON b.ad_id = t.ad_id
)
-- SELECT app_or_linear
-- , station_type
-- , ad_id
SELECT ad_id
, -SUM(CASE WHEN p_dma>0 THEN p_dma * LOG(p_dma) ELSE 0 END) AS entropy
, MAX(k_dmas) AS k_dmas
, CASE WHEN MAX(k_dmas) > 1 THEN (-SUM(CASE WHEN p_dma>0 THEN p_dma * LOG(p_dma) ELSE 0 END)) / LOG(MAX(k_dmas)) ELSE 0.0 END AS entropy_norm
FROM joined
GROUP BY 1--,2,3;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_entropy_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_entropy_unbinned AS
-- 1) DMA universe per slice (this is what adds the "0s")
WITH dma_universe AS (
  SELECT fk_dma_id
  , active_tvs
  FROM dev.mohit_gangwani.ad_labeling_opportunities_unbinned
)
-- 2) Ad universe per slice
, ad_universe AS (
  SELECT DISTINCT ad_id
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
)
-- 3) Full grid: every (ad, slice) × every DMA in that slice
, grid AS (
  SELECT a.ad_id
  , d.fk_dma_id
  , d.active_tvs
  FROM ad_universe a, dma_universe d
)
-- 4) Left-join impressions; missing DMAs become 0 impressions
, joined AS (
  SELECT g.ad_id
  , g.fk_dma_id
  , g.active_tvs
  , COALESCE(i.impressions, 0) AS impressions
  FROM grid g
  LEFT JOIN dev.mohit_gangwani.ad_labeling_impression_count_unbinned i
    ON g.ad_id = i.ad_id
   AND g.fk_dma_id = i.fk_dma_id
)
-- 5) Convert to TV-normalized "intensity" (imps per active TV)
, rates AS (
  SELECT ad_id
  , fk_dma_id
  , CASE WHEN active_tvs > 0 THEN impressions * 1.0 / active_tvs
         ELSE 0.0
    END AS rate_per_tv
  FROM joined
)
-- 6) Total mass per (ad, slice) for probability normalization
, tot AS (
  SELECT ad_id
  , SUM(rate_per_tv) AS total_rate
  , COUNT(*) AS k_total_dmas
  FROM rates
  GROUP BY 1
)
-- 7) Probability distribution over ALL DMAs (includes zeros)
, p AS (
  SELECT r.ad_id
  , r.fk_dma_id
  , t.k_total_dmas
  , CASE WHEN t.total_rate > 0 THEN r.rate_per_tv / t.total_rate
         ELSE 0.0
    END AS p_dma
  FROM rates r
  JOIN tot t
    ON r.ad_id = t.ad_id
)
SELECT ad_id
, MAX(k_total_dmas) AS k_total_dmas
, -SUM(CASE WHEN p_dma > 0 THEN p_dma * LOG(p_dma) ELSE 0 END) AS entropy_tv_norm_full
, CASE WHEN MAX(k_total_dmas) > 1 THEN (-SUM(CASE WHEN p_dma > 0 THEN p_dma * LOG(p_dma) ELSE 0 END)) / LOG(MAX(k_total_dmas))
       ELSE 0.0
  END AS entropy_norm
FROM p
GROUP BY 1;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_entropy_unbinned
WHERE ad_id = 'AE16546-2025-42-03564'
ORDER BY entropy_norm
LIMIT 100

In [0]:
%sql
-- significant_dma_count_05 = count of DMAs where share >= 0.05
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned AS
WITH base AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
),
tot AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , SUM(impressions) * 1.0 AS tot_imp
  FROM base
  GROUP BY 1--,2,3
),
shares AS (
  -- SELECT b.app_or_linear
  -- , b.station_type
  -- , b.ad_id
  SELECT b.ad_id
  , b.fk_dma_id
  , b.impressions * 1.0 / t.tot_imp AS dma_share
  FROM base b
  JOIN tot t
  --   ON b.app_or_linear = t.app_or_linear
  --  AND b.station_type = t.station_type
  --  AND b.ad_id = t.ad_id
  ON b.ad_id = t.ad_id
)
-- SELECT app_or_linear
-- , station_type
-- , ad_id
SELECT ad_id
, SUM(CASE WHEN dma_share >= 0.05 THEN 1 ELSE 0 END) AS significant_dma_count_05
FROM shares
GROUP BY 1--,2,3;

In [0]:
%sql
WITH base AS (
  SELECT ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
  WHERE ad_id IN ('AE16546-2025-23-00849','AE17151-2021-46-00240','AE16546-2024-44-06302','AE16546-2025-01-06980','AE16546-2024-50-08087','AE16546-2025-37-10553','AE16546-2025-28-09574','AE16546-2025-41-11821','AE16546-2024-48-05245','AE16546-2025-49-00080','AE16546-2024-14-04396','AE16546-2024-45-01382','AE16546-2025-49-02205')
),
tot AS (
  SELECT ad_id
  , SUM(impressions) * 1.0 AS tot_imp
  FROM base
  GROUP BY 1--,2,3
)
SELECT b.ad_id
, b.fk_dma_id
, b.impressions * 1.0 / t.tot_imp AS dma_share
FROM base b
JOIN tot t
ON b.ad_id = t.ad_id
ORDER BY 1, 3 DESC

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned
WHERE ad_id = 'AE16546-2025-50-00743'
ORDER BY significant_dma_count_05 DESC
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned AS
WITH base AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
),
ranked AS (
  SELECT *
  -- , ROW_NUMBER() OVER (PARTITION BY app_or_linear, station_type, ad_id ORDER BY impressions DESC) AS rn
  , ROW_NUMBER() OVER (PARTITION BY ad_id ORDER BY impressions DESC) AS rn
  FROM base
),
tot AS (
  -- SELECT app_or_linear
  -- , station_type
  SELECT ad_id
  , ad_id
  , SUM(impressions) * 1.0 AS tot_imp
  FROM base
  GROUP BY 1--, 2, 3
),
top5 AS (
  -- SELECT app_or_linear
  -- , station_type
  -- , ad_id
  SELECT ad_id
  , SUM(impressions) * 1.0 AS top5_imp
  FROM ranked
  WHERE rn <= 5
  GROUP BY 1--, 2, 3
)
-- SELECT t.app_or_linear
-- , t.station_type
-- , t.ad_id
SELECT t.ad_id
, CASE WHEN t.tot_imp > 0 THEN top5.top5_imp / t.tot_imp ELSE 0.0
  END AS top5_dma_mix_ratio
FROM tot t
JOIN top5
--   ON t.app_or_linear = top5.app_or_linear
--  AND t.station_type = top5.station_type
--  AND t.ad_id = top5.ad_id;
 ON t.ad_id = top5.ad_id;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned
WHERE ad_id = 'AE16546-2025-50-00743'
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_final_features_unbinned_011326;
CREATE TABLE dev.mohit_gangwani.ad_labeling_final_features_unbinned_011326 AS
WITH slice_features AS (
  SELECT t.ad_id
  , t.total_impressions
  , t.total_opportunities
  , c.coverage_score
  , e.entropy_norm
  , s.significant_dma_count_05
  , o.dma_in_95
  , m.top5_dma_mix_ratio
  , i.total_states
  FROM (
    SELECT ad_id
    , SUM(impressions) AS total_impressions
    , SUM(opportunities) AS total_opportunities
    FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
    GROUP BY 1
  ) t
  JOIN dev.mohit_gangwani.ad_labeling_ad_impression_count_unbinned i
    ON t.ad_id = i.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_coverage_unbinned c
    ON t.ad_id = c.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_entropy_unbinned e
    ON t.ad_id = e.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned s
    ON t.ad_id = s.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned m
    ON t.ad_id = m.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_over_95_unbinned o
    ON t.ad_id = o.ad_id
)
SELECT ad_id
, SUM(total_impressions) AS total_impressions
, SUM(total_opportunities) AS total_opportunities
, SUM(COALESCE(coverage_score, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS coverage_score
, SUM(COALESCE(entropy_norm, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS entropy_norm
, SUM(COALESCE(top5_dma_mix_ratio, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS top5_dma_mix_ratio
, MAX(COALESCE(significant_dma_count_05, 0.0)) AS significant_dma_count_05
, MAX(COALESCE(dma_in_95, 0.0)) AS dma_in_90
, MAX(COALESCE(total_states, 0.0)) AS total_states
FROM slice_features
GROUP BY ad_id

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_final_features_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_final_features_unbinned AS
WITH slice_features AS (
  SELECT t.app_or_linear
  , t.station_type
  , t.ad_id
  , t.total_impressions
  , t.total_opportunities
  , c.coverage_score
  , e.entropy_norm
  , s.significant_dma_count_05
  , m.top5_dma_mix_ratio
  FROM (
    SELECT app_or_linear
    , station_type
    , ad_id
    , SUM(impressions) AS total_impressions
    , SUM(opportunities) AS total_opportunities
    FROM dev.mohit_gangwani.ad_labeling_phat_unbinned
    GROUP BY 1,2,3
  ) t
  LEFT JOIN dev.mohit_gangwani.ad_labeling_coverage_unbinned c
    ON t.app_or_linear = c.app_or_linear
   AND t.station_type = c.station_type
   AND t.ad_id = c.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_entropy_unbinned e
    ON t.app_or_linear = e.app_or_linear
   AND t.station_type = e.station_type
   AND t.ad_id = e.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned s
    ON t.app_or_linear = s.app_or_linear
   AND t.station_type = s.station_type
   AND t.ad_id = s.ad_id
  LEFT JOIN dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned m
    ON t.app_or_linear = m.app_or_linear
   AND t.station_type = m.station_type
   AND t.ad_id = m.ad_id
)
SELECT ad_id
, SUM(total_impressions) AS total_impressions
, SUM(total_opportunities) AS total_opportunities
, SUM(COALESCE(coverage_score, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS coverage_score
, SUM(COALESCE(entropy_norm, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS entropy_norm
, SUM(COALESCE(top5_dma_mix_ratio, 0.0) * total_impressions) / NULLIF(SUM(total_impressions), 0.0) AS top5_dma_mix_ratio
, MAX(COALESCE(significant_dma_count_05, 0.0)) AS significant_dma_count_05
, COUNT(*) AS num_slices
FROM slice_features
GROUP BY ad_id

In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.ad_labeling_final_features_unbinned

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_final_features_unbinned
WHERE ad_id = 'AE16546-2025-50-00743'
LIMIT 100

In [0]:
%sql
SELECT * FROM prod.detection.commercial_id_external_firehose WHERE external_id = 'AE16546-2024-06-02313'

In [0]:
%sql
SELECT dma.dma_name, COUNT(*)
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table x
JOIN prod.detection.dma ON dma.dma_id = x.fk_dma_id
WHERE ad_id IN ('AE16546-2024-06-02313')
GROUP BY 1

In [0]:
%sql
SELECT MIN(session_start), MAX(session_end), COUNT(*), COUNT(DISTINCT fk_tvid) FROM dev.carol_zhou.superbowl_vcf_hourly_final_de
LIMIT 100

In [0]:
%sql
SELECT MIN(session_start), MAX(session_end), COUNT(*), COUNT(DISTINCT fk_tvid) FROM dev.carol_zhou.superbowl_vcf_hourly_final_de
LIMIT 100